# Visualización del conjunto de datos

En este notebook se muestran algunos de los mecanismos más utilizados para la visualización del conjunto de datos.

## Conjunto de datos

### Descripción
NSL-KDD es un conjunto de datos propuesto para resolver algunos de los problemas inherentes del conjunto de datos KDD'99. Sigue siendo un benchmark efectivo para comparar métodos de detección de intrusiones.

### Ficheros de datos utilizados
* **KDDTrain+.csv**: Conjunto de entrenamiento completo NSL-KDD en formato CSV
* **KDDTest+.csv**: Conjunto de pruebas completo NSL-KDD en formato CSV

Los archivos CSV no traen los nombres de las características. Para ello se obtuvieron del paper del dataset:
_M. Tavallaee, E. Bagheri, W. Lu, and A. Ghorbani, "A Detailed Analysis of the KDD CUP 99 Data Set," CISDA, 2009._

## 1. Lectura del conjunto de datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'difficulty'
]


def load_nsl_kdd_csv(data_path):
    """Carga un archivo CSV del dataset NSL-KDD y asigna nombres de columnas."""
    return pd.read_csv(data_path, header=None, names=COLUMN_NAMES)

In [ ]:
# Lectura del conjunto de datos mediante funciones básicas de Python
with open("NSL_KDD-master/KDDTrain+.csv", "r") as train_set:
    lineas = train_set.readlines()

print(f"Número de líneas leídas: {len(lineas)}")
lineas[:3]

In [ ]:
# Lectura del conjunto de datos utilizando la función definida
df_train = load_nsl_kdd_csv("NSL_KDD-master/KDDTrain+.csv")
df_test  = load_nsl_kdd_csv("NSL_KDD-master/KDDTest+.csv")

print("Forma del training set:", df_train.shape)
print("Forma del test set:",     df_test.shape)

In [ ]:
df_train.head()

## 2. Funciones básicas de visualización de los datos

* El proceso de visualización siempre debe realizarse sobre el training set y apartando el test set. Esto evita que nuestro cerebro genere intuiciones del test set que podemos incorporar en nuestro modelo.
* Una buena práctica es crear una copia del training set y jugar con ella. De esta manera, si realizamos transformaciones que dañan el training set, el original no se ve afectado.

In [ ]:
# Para el resto del notebook trabajaremos con una copia del training set
df = df_train.copy()
df.head()

In [ ]:
# Mostrar en pantalla un número determinado de filas
df.head(10)

In [ ]:
# Mostrar información básica sobre el conjunto de datos
df.info()

In [ ]:
# Tipos de datos detectados por Pandas
df.dtypes

In [ ]:
# Mostrar información estadística sobre el conjunto de datos
df.describe()

### Significado de los valores de `describe()`

- **count**: Cantidad de datos no nulos. Sirve para detectar datos faltantes.
- **25% (Q1)**: Valor por debajo del cual está el 25% de los datos.
- **50% (Mediana)**: Valor central. Más robusto que la media frente a outliers.
- **75% (Q3)**: Valor por debajo del cual está el 75% de los datos.

```
mín ---- Q1 ---- mediana ---- Q3 ---- máx
```

**IQR = Q3 - Q1** → Valores muy fuera de este rango son posibles outliers.

In [ ]:
# Mostrar los valores únicos que tiene un atributo determinado
df["protocol_type"].value_counts()

In [ ]:
# Mostrar la distribución de la clase objetivo
df["class"].value_counts()

In [ ]:
# Representación gráfica de una característica categórica
%matplotlib inline

df["protocol_type"].value_counts().plot(kind="bar")
plt.title("Distribución de protocol_type")
plt.xlabel("protocol_type")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# Representar gráficamente la distribución de los atributos
df.hist(bins=50, figsize=(20, 15))
plt.show()

## 3. Funciones avanzadas de visualización de los datos

### Buscando correlaciones

* Se puede calcular el coeficiente de correlación estándar para ver la correlación entre cada par de atributos.
* El coeficiente de correlación solo mide **correlaciones lineales**.
* Para calcular correlaciones, primero hay que convertir a valores numéricos la variable objetivo `class` y los atributos categóricos.

In [ ]:
# El atributo class de nuestro conjunto de datos tiene valores categóricos
df["class"]

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Transformamos los valores del atributo class de categóricos a numéricos
labelencoder = LabelEncoder()
df["class"] = labelencoder.fit_transform(df["class"])

# Transformamos los valores de los atributos categóricos a numéricos
df["protocol_type"] = labelencoder.fit_transform(df["protocol_type"])
df["service"] = labelencoder.fit_transform(df["service"])
df["flag"] = labelencoder.fit_transform(df["flag"])

df

In [ ]:
# Mostrar la correlación entre los atributos del conjunto de datos
corr_matrix = df.corr(numeric_only=True)
corr_matrix["class"].sort_values(ascending=False)

In [ ]:
# Mostrar correlación lineal entre todos los atributos del conjunto de datos
df.corr(numeric_only=True)

In [ ]:
# Representar gráficamente la matriz de correlación
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 8))
ax.matshow(corr)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.show()

In [ ]:
# Representar gráficamente las correlaciones entre atributos seleccionados
from pandas.plotting import scatter_matrix

attributes = ["same_srv_rate", "dst_host_srv_count", "class", "dst_host_same_srv_rate"]

scatter_matrix(df[attributes], figsize=(12, 8))
plt.show()

In [ ]:
df.corr(numeric_only=True)["class"].sort_values(ascending=False)